In [0]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from contextlib import contextmanager

def get_data(sql_text, params):
    df = spark.sql(sql_text, args=params)  

    return df

def plot_did_percent_change_three_comparisons(
    df,
    *,
    period_col="period",         
    window_col="window",        
    treat_period="CY",
    control_period="LY",
    comparisons=(("PRE","COMM","Pre vs Comm"),
                 ("PRE","POST","Pre vs Post rollout"),
                 ("COMM","POST","Comm vs Post rollout")),
    metrics=None,
    metric_pretty=None,
    metric_group=None,
    group_order=None,
    eps=1e-12,                
    outfile=None,
    figsize=(12, 6),

    # --- NEW: right-strip text settings ---
    group_question=None,
    strip_left=0.84,
    strip_width=0.16,
    strip_fontsize=8,
):
    """
    Computes DID % change and plots it in the same 3-column chart.

    For each metric m and comparison (A -> B):
      pct_change(period) = (B - A) / (A + eps)
      DID_pct = pct_change(treat_period) - pct_change(control_period)
    """

    if metrics is None:
        exclude = {period_col, window_col, "country_name"}
        metrics = [c for c in df.columns if c not in exclude]

    if metric_pretty is None:
        metric_pretty = {m: m for m in metrics}

    if metric_group is None:
        metric_group = {m: "Metrics" for m in metrics}

    if group_order is None:
        seen = []
        for m in metrics:
            g = metric_group.get(m, "Metrics")
            if g not in seen:
                seen.append(g)
        group_order = seen

    if group_question is None:
        group_question = {
            "Extensive Margin": (
                "Extensive margin:\n"
                "Did suppliers or tours \n"
                "enter/exit the platform after the change?"
            ),
            "Intensive Margin": (
                "Intensive margin:\n"
                "Among suppliers/tours that \n"
                "stayed, did supply intensity\n"
                "change \n"
                "(e.g availability, capacity)?"
            ),
            "Customer Response": (
                "Customer response:\n"
                "Did demand shift after\n"
                "the change?"
            ),
        }

    # Validation: need required windows per period
    needed_windows = {w for a, b, _ in comparisons for w in (a, b)}
    for p in (treat_period, control_period):
        sub = df[df[period_col] == p]
        present = set(sub[window_col].unique())
        missing = needed_windows - present
        if missing:
            raise ValueError(f"Missing windows for period={p}: {sorted(missing)}")

    treat = df[df[period_col] == treat_period].set_index(window_col)
    ctrl  = df[df[period_col] == control_period].set_index(window_col)

    # --- Compute DID % change ---
    did_rows = []
    for m in metrics:
        for a, b, label in comparisons:
            treat_pct = (treat.loc[b, m] - treat.loc[a, m]) / (treat.loc[a, m] + eps)
            ctrl_pct  = (ctrl.loc[b, m]  - ctrl.loc[a, m])  / (ctrl.loc[a, m]  + eps)
            did = treat_pct - ctrl_pct
            did_rows.append({
                "metric": m,
                "comparison": label,
                "did_pct": did,
                "group": metric_group.get(m, "Metrics"),
                "pretty": metric_pretty.get(m, m),
            })

    did_df = pd.DataFrame(did_rows)

    # --- Y order: group blocks, then metrics within group ---
    ordered_metrics = []
    for g in group_order:
        ordered_metrics.extend([m for m in metrics if metric_group.get(m, "Metrics") == g])

    y_map = {m: i for i, m in enumerate(ordered_metrics)}
    y_labels = [metric_pretty.get(m, m) for m in ordered_metrics]

    # --- Plot ---
    fig, axes = plt.subplots(1, 3, figsize=figsize, sharey=True)

    for ax, (_, _, title) in zip(axes, comparisons):
        sub = did_df[did_df["comparison"] == title].set_index("metric")
        xs = [sub.loc[m, "did_pct"] for m in ordered_metrics]
        ys = [y_map[m] for m in ordered_metrics]

        ax.scatter(xs, ys, s=35, color="black", zorder=3)
        ax.axvline(0, color="red", linestyle="--", linewidth=1.5, zorder=2)
        ax.grid(True, axis="x", color="0.9")
        ax.set_title(title, fontsize=11)

    axes[0].set_yticks(range(len(ordered_metrics)))
    axes[0].set_yticklabels(y_labels)
    axes[0].invert_yaxis()

    # --- Group spans + separators ---
    group_spans = []
    start = 0
    for g in group_order:
        ms = [m for m in ordered_metrics if metric_group.get(m, "Metrics") == g]
        if not ms:
            continue
        end = start + len(ms) - 1
        group_spans.append((g, start, end))
        start = end + 1

    for ax in axes:
        for _, s, e in group_spans[:-1]:
            ax.axhline(e + 0.5, color="0.85", linewidth=2)

    # --- Right-side category strip (WIDER + wrapped question text) ---
    strip_ax = fig.add_axes([strip_left, 0.12, strip_width, 0.78])
    strip_ax.set_xlim(0, 1)
    strip_ax.set_ylim(-0.5, len(ordered_metrics) - 0.5)
    strip_ax.invert_yaxis()
    strip_ax.axis("off")

    for g, s, e in group_spans:
        rect = Rectangle((0, s - 0.5), 1, (e - s + 1), facecolor="0.9", edgecolor="0.2")
        strip_ax.add_patch(rect)

        label = group_question.get(g, g)
        strip_ax.text(
            0.5, (s + e) / 2,
            label,
            ha="center", va="center",
            fontsize=strip_fontsize,
            linespacing=1.15,
            wrap=True,
            clip_on=True,
        )

    fig.supxlabel(
        f"DID % change = (({treat_period} (B−A)/A) − ({control_period} (B−A)/A))",
        fontsize=11
    )

    fig.tight_layout(rect=[0.05, 0.05, strip_left - 0.01, 0.95])

    if outfile:
        plt.savefig(outfile, dpi=200, bbox_inches="tight")

    return fig, axes, did_df

In [0]:
# Params 
params = {
    "comm_date": "2026-02-01",
    "rollout_date": "2026-02-10",
    "start_date": "2026-01-01",
    "end_date": "2026-03-01",
    "country": "Italy",
    "tour_ids": "62214,153354,195566,562279",
    "x_weeks": 4
}

# bookings_sql = Path("/Workspace/Users/shazeb.asad@getyourguide.com/Pricing-Supply-Analytics/DST_Analysis/SQL/Aggregated/bookings_agg.sql").read_text()

# supply_sql = Path("/Workspace/Users/shazeb.asad@getyourguide.com/Pricing-Supply-Analytics/DST_Analysis/SQL/Aggregated/supply_availability_agg.sql").read_text()

customer_sql = Path("/Workspace/Users/shazeb.asad@getyourguide.com/Pricing-Supply-Analytics/DST_Analysis/SQL/Aggregated/customer_metrics_agg.sql").read_text()

price_index_sql = Path("/Workspace/Users/shazeb.asad@getyourguide.com/Pricing-Supply-Analytics/DST_Analysis/SQL/Aggregated/price_index_agg.sql").read_text()

# bookings_df = get_data(params, bookings_sql)
# print(bookings_df.head())

# supply_df = get_data(params, supply_sql)
# print(supply_df.head())

# customer_df = get_data(params, customer_sql)
# print(customer_df.head())

price_df = get_data(price_index_sql,params)
print(price_df.head())

# supply_bookings_df = supply_df.merge(bookings_df, on= ["period","window"], how = "inner")

# add_price_index = supply_bookings_df.merge(price_df, on= ["period","window"], how = "inner")

#combined_df = add_price_index.merge(customer_df, on= ["period","window"], how = "inner")



In [0]:
# Convert Decimal columns to float to avoid type errors
combined_df_numeric = combined_df.copy()
for col in combined_df_numeric.columns:
    if combined_df_numeric[col].dtype == 'object':
        try:
            combined_df_numeric[col] = combined_df_numeric[col].astype(float)
        except (ValueError, TypeError):
            pass  # Keep as-is if conversion fails

fig, axes, did_df = plot_did_percent_change_three_comparisons(
    combined_df_numeric,
    metrics=[
        "total_tours",
        "active_tours","share_active_tours",
        "total_suppliers","active_suppliers","share_active_suppliers",
        "avg_days_online_per_tour","avg_days_online_per_active_tour",
        "avg_days_online_per_supplier","avg_days_online_per_active_supplier",
        "bookings", "tickets", "nr", "gmv"
    ],
    metric_pretty={
        "total_tours":"Total tours",
        "active_tours":"Active tours",
        "share_active_tours":"Share active tours",
        "total_suppliers":"Total suppliers",
        "active_suppliers":"Active suppliers",
        "share_active_suppliers":"Share active suppliers",
        "avg_days_online_per_tour":"Avg days online / tour",
        "avg_days_online_per_active_tour":"Avg days online / active tour",
        "avg_days_online_per_supplier":"Avg days online / supplier",
        "avg_days_online_per_active_supplier":"Avg days online / active supplier",
        "bookings": "Bookings", 
        "tickets": "Tickets", 
        "nr": "Net Revenue", 
        "gmv": "GMV"
    },
    metric_group={
        "total_tours":"Extensive Margin",
        "active_tours":"Extensive Margin",
        "share_active_tours":"Extensive Margin",
        "total_suppliers":"Extensive Margin",
        "active_suppliers":"Extensive Margin",
        "share_active_suppliers":"Extensive Margin",
        "avg_days_online_per_tour":"Intensive Margin",
        "avg_days_online_per_active_tour":"Intensive Margin",
        "avg_days_online_per_supplier":"Intensive Margin",
        "avg_days_online_per_active_supplier":"Intensive Margin",
        "bookings": "Customer Response",
        "tickets": "Customer Response",
        "nr": "Customer Response",
        "gmv": "Customer Response"
    },
    group_order=["Extensive Margin","Intensive Margin","Customer Response"],
    strip_width = 0.2, strip_fontsize = 10)

plt.show()

In [0]:

# # Read SQL file (must be a single statement, typically a SELECT with :named_params)
# sql_text = Path("/Workspace/Users/shazeb.asad@getyourguide.com/Pricing-Supply-Analytics/DST_Analysis/SQL/supply_availability.sql").read_text()

# # Params you want to inject
# params = {
#     "comm_date": "2026-02-01",
#     "rollout_date": "2026-02-10",
#     "start_date": "2026-01-01",
#     "end_date": "2026-03-01",
#     "country": "Italy",
#     "tour_ids_csv": "62214,153354,195566,562279",
# }

# # Execute and get a DataFrame (no table write)
# df = spark.sql(sql_text, args=params)

# display(df)           # in a Databricks notebook
# # df.show(20, False)  # alternative


In [0]:
# from pathlib import Path

# # Read SQL file
# sql_text = Path("/Workspace/Users/shazeb.asad@getyourguide.com/Pricing-Supply-Analytics/DST_Analysis/SQL/Aggregated/bookings_agg.sql").read_text()

# # Params 
# params = {
#     "comm_date": "2026-02-01",
#     "rollout_date": "2026-02-10",
#     "start_date": "2026-01-01",
#     "end_date": "2026-03-01",
#     "country": "Italy",
#     "tour_ids": "62214,153354,195566,562279",
#     "x_weeks": 4
# }

# df = spark.sql(sql_text, args=params).toPandas()
# display(df)  

In [0]:
# from pathlib import Path

# # Read SQL file
# sql_text = Path("/Workspace/Users/shazeb.asad@getyourguide.com/Pricing-Supply-Analytics/DST_Analysis/SQL/Aggregated/supply_availability_agg.sql").read_text()

# # Params 
# params = {
#     "comm_date": "2026-02-01",
#     "rollout_date": "2026-02-10",
#     "start_date": "2026-01-01",
#     "end_date": "2026-03-01",
#     "country": "Italy",
#     "tour_ids": "62214,153354,195566,562279",
#     "x_weeks": 4
# }

# df = spark.sql(sql_text, args=params).toPandas()
# display(df)           



In [0]:
# import pandas as pd
# import matplotlib.pyplot as plt
# from matplotlib.patches import Rectangle

# def plot_did_percent_change_three_comparisons(
#     df,
#     *,
#     period_col="period",          # e.g. "CY", "LY"
#     window_col="window",          # e.g. "PRE", "COMM", "POST"
#     treat_period="CY",
#     control_period="LY",
#     comparisons=(("PRE","COMM","Pre vs Comm"),
#                  ("PRE","POST","Pre vs Post rollout"),
#                  ("COMM","POST","Comm vs Post rollout")),
#     metrics=None,
#     metric_pretty=None,
#     metric_group=None,
#     group_order=None,
#     eps=1e-12,                    # guard for divide-by-zero
#     outfile=None,
#     figsize=(12, 6),
# ):
#     """
#     Computes DID % change and plots it in the same 3-column chart.

#     For each metric m and comparison (A -> B):
#       pct_change(period) = (B - A) / (A + eps)
#       DID_pct = pct_change(treat_period) - pct_change(control_period)

#     X-axis is DID % (not log points). Example: 0.10 = +10 percentage points (relative change).
#     """

#     if metrics is None:
#         # default: all numeric cols except common dims
#         exclude = {period_col, window_col, "country_name"}
#         metrics = [c for c in df.columns if c not in exclude]

#     if metric_pretty is None:
#         metric_pretty = {m: m for m in metrics}

#     if metric_group is None:
#         metric_group = {m: "Metrics" for m in metrics}

#     if group_order is None:
#         seen = []
#         for m in metrics:
#             g = metric_group.get(m, "Metrics")
#             if g not in seen:
#                 seen.append(g)
#         group_order = seen

#     # Basic validation: need one row per (period, window) to index cleanly
#     needed_windows = {w for a, b, _ in comparisons for w in (a, b)}
#     for p in (treat_period, control_period):
#         sub = df[df[period_col] == p]
#         present = set(sub[window_col].unique())
#         missing = needed_windows - present
#         if missing:
#             raise ValueError(f"Missing windows for period={p}: {sorted(missing)}")

#     treat = df[df[period_col] == treat_period].set_index(window_col)
#     ctrl  = df[df[period_col] == control_period].set_index(window_col)

#     # --- Compute DID % change ---
#     did_rows = []
#     for m in metrics:
#         for a, b, label in comparisons:
#             treat_pct = (treat.loc[b, m] - treat.loc[a, m]) / (treat.loc[a, m] + eps)
#             ctrl_pct  = (ctrl.loc[b, m]  - ctrl.loc[a, m])  / (ctrl.loc[a, m]  + eps)
#             did = treat_pct - ctrl_pct
#             did_rows.append({
#                 "metric": m,
#                 "comparison": label,
#                 "did_pct": did,                 # e.g. 0.12 = +12%
#                 "group": metric_group.get(m, "Metrics"),
#                 "pretty": metric_pretty.get(m, m),
#             })

#     did_df = pd.DataFrame(did_rows)

#     # --- Y order: group blocks, then metrics within group ---
#     ordered_metrics = []
#     for g in group_order:
#         ordered_metrics.extend([m for m in metrics if metric_group.get(m, "Metrics") == g])

#     y_map = {m: i for i, m in enumerate(ordered_metrics)}
#     y_labels = [metric_pretty.get(m, m) for m in ordered_metrics]

#     # --- Plot (one dot per metric: the DID) ---
#     fig, axes = plt.subplots(1, 3, figsize=figsize, sharey=True)

#     for ax, (_, _, title) in zip(axes, comparisons):
#         sub = did_df[did_df["comparison"] == title].set_index("metric")

#         xs = [sub.loc[m, "did_pct"] for m in ordered_metrics]
#         ys = [y_map[m] for m in ordered_metrics]

#         ax.scatter(xs, ys, s=35, color="black", zorder=3)
#         ax.axvline(0, color="red", linestyle="--", linewidth=1.5, zorder=2)
#         ax.grid(True, axis="x", color="0.9")
#         ax.set_title(title, fontsize=11)

#     axes[0].set_yticks(range(len(ordered_metrics)))
#     axes[0].set_yticklabels(y_labels)
#     axes[0].invert_yaxis()

#     # --- Group separators ---
#     group_spans = []
#     start = 0
#     for g in group_order:
#         ms = [m for m in ordered_metrics if metric_group.get(m, "Metrics") == g]
#         if not ms:
#             continue
#         end = start + len(ms) - 1
#         group_spans.append((g, start, end))
#         start = end + 1

#     for ax in axes:
#         for _, s, e in group_spans[:-1]:
#             ax.axhline(e + 0.5, color="0.85", linewidth=2)

#     # --- Right-side category strip ---
#     strip_ax = fig.add_axes([0.88, 0.12, 0.10, 0.78])
#     strip_ax.set_xlim(0, 1)
#     strip_ax.set_ylim(-0.5, len(ordered_metrics) - 0.5)
#     strip_ax.invert_yaxis()
#     strip_ax.axis("off")

#     for g, s, e in group_spans:
#         rect = Rectangle((0, s - 0.5), 1, (e - s + 1), facecolor="0.9", edgecolor="0.2")
#         strip_ax.add_patch(rect)
#         strip_ax.text(0.5, (s + e) / 2, g, ha="center", va="center", fontsize=11)

#     fig.supxlabel(
#         f"DID % change = (({treat_period} (B−A)/A) − ({control_period} (B−A)/A))",
#         fontsize=11
#     )
#     fig.tight_layout(rect=[0.05, 0.05, 0.87, 0.95])

#     if outfile:
#         plt.savefig(outfile, dpi=200, bbox_inches="tight")

#     return fig, axes, did_df


# # -----------------------------
# # Example usage:
# fig, axes, did_df = plot_did_percent_change_three_comparisons(
#     df,
#     metrics=[
#         "total_tours","active_tours","share_active_tours",
#         "total_suppliers","active_suppliers","share_active_suppliers",
#         "avg_days_online_per_tour","avg_days_online_per_active_tour",
#         "avg_days_online_per_supplier","avg_days_online_per_active_supplier",
#     ],
#     metric_pretty={
#         "total_tours":"Total tours",
#         "active_tours":"Active tours",
#         "share_active_tours":"Share active tours",
#         "total_suppliers":"Total suppliers",
#         "active_suppliers":"Active suppliers",
#         "share_active_suppliers":"Share active suppliers",
#         "avg_days_online_per_tour":"Avg days online / tour",
#         "avg_days_online_per_active_tour":"Avg days online / active tour",
#         "avg_days_online_per_supplier":"Avg days online / supplier",
#         "avg_days_online_per_active_supplier":"Avg days online / active supplier",
#     },
#     metric_group={
#         "total_tours":"Extensive Margin",
#         "active_tours":"Extensive Margin",
#         "share_active_tours":"Extensive Margin",
#         "total_suppliers":"Extensive Margin",
#         "active_suppliers":"Extensive Margin",
#         "share_active_suppliers":"Extensive Margin",
#         "avg_days_online_per_tour":"Intensive Margin",
#         "avg_days_online_per_active_tour":"Intensive Margin",
#         "avg_days_online_per_supplier":"Intensive Margin",
#         "avg_days_online_per_active_supplier":"Intensive Margin",
#     },
#     group_order=["Extensive Margin","Intensive Margin"],
# )
# plt.show()
# #
# # did_df  # table of DID % changes